<a href="https://colab.research.google.com/github/hashirama21/gik-icechain/blob/main/notebooks/gik_icechain_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GIK-IceChain - Live Walkthrough (MinIO, real execution)

Runs the pipeline **component by component on real data from the live MinIO store**
(no synthetic fallback), then the full pipeline, then the three checks that close the
proposal's open promises.

1. **Setup** - locate repo, read MinIO credentials, download public prerequisites.
2. **C1** - open the real IceChunk store; read a real forecast day's `tp`.
3. **C2** - compute real exceedance on that forecast, and read the live exceedance Zarr.
4. **C3** - real CRMA risk for admin-1 units from the live exceedance store.
5. **Full pipeline** - `gik-icechain run-all` against MinIO.
6. **Benchmarks** - live `run_benchmark`, scored against the proposal's targets.
7. **Archive coverage** - the full ~1200-day corpus, and the 0.4/0.25 deg grid change.
8. **Innovation 4** - AIFS vs IFS parallel track.
9. **Innovation 2** - adaptive vs static GEV thresholds (the missing ablation).

Sections 1-6 need MinIO credentials (`MINIO_ENDPOINT_URL` / `MINIO_ACCESS_KEY` /
`MINIO_SECRET_KEY`, via env, Colab secrets, or `.env`) and do **not** fall back to
synthetic data - if the store is unreachable, the cells raise.

Section 7 needs no credentials. Sections 8-9 are heavy (hours of compute, and 9 needs
the GPM archive), so they are **opt-in**: set `NB_RUN_AIFS=1` / `NB_RUN_ABLATION=1`.


## 0. Setup - repo, MinIO credentials, prerequisites

In [ ]:
import os, sys, subprocess
from pathlib import Path

def _bootstrap() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)
    return repo

REPO = _bootstrap()
DATA = REPO / "data"
CONFIG = "configs/default.yaml"
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Sections 4-6 (run-all, EM-DAT validation, benchmarks) take ~40 min:
# NB_RUN_FULL=0 keeps the notebook CI-sized (sections 0-3 and 7 only).
RUN_FULL = os.getenv("NB_RUN_FULL", "1") == "1"

def _secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, default)

_minio = _secret("MINIO") or _secret("MINIO_ENDPOINT_URL")
ENDPOINT = _minio if _minio.startswith("http") else (f"http://{_minio}" if _minio else "")
KEY = _secret("MINIO_ACCESS_KEY") or _secret("AWS_ACCESS_KEY_ID")
SECRET = _secret("MINIO_SECRET_KEY") or _secret("AWS_SECRET_ACCESS_KEY")
if not (ENDPOINT and KEY and SECRET):
    raise RuntimeError(
        "MinIO credentials required (MINIO_ENDPOINT_URL/MINIO_ACCESS_KEY/MINIO_SECRET_KEY). "
        "This notebook runs against the live store - no synthetic fallback.")

os.environ["AWS_ENDPOINT_URL"] = ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"] = KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET
os.environ.setdefault("AWS_REGION", "eu-west-1")
os.environ.setdefault("ECCODES_PYTHON_USE_FINDLIBS", "1")

STORAGE_OPTIONS = {"endpoint_url": ENDPOINT}

tools_script = str(REPO / "scripts" / "tools.py")
subprocess.run([sys.executable, tools_script, "download", "--component", "all"],
               cwd=str(REPO), check=True)
subprocess.run([sys.executable, tools_script, "download-thresholds"],
               cwd=str(REPO), check=True)

from gik_icechain.shared.config import load_config
cfg = load_config(REPO / CONFIG)

# ECMWF latitude is stored descending: slice(north, south).
_b = cfg.component2.spatial.bbox  # [lat_min, lat_max, lon_min, lon_max]
BBOX = {cfg.component2.spatial.lat_dim: slice(_b[1], _b[0]),
        cfg.component2.spatial.lon_dim: slice(_b[2], _b[3])}

print("Repo:", REPO)
print("Full run:", RUN_FULL)
print("Thresholds check:", "OK" if (DATA / "cmorph_thresholds").exists() else "MISSING")

## 1. Component 1 - real IceChunk store

Open the live store, list its committed days, and read one real forecast day's `tp`
(subset to the bbox so the GRIB decode is fast). No synthetic data.

In [ ]:
from gik_icechain.conversion.icechunk_writer import IceChainStore
from gik_icechain.shared.grid import lat_lon_res

store = IceChainStore(cfg.outputs.icechunk_store_uri,
                      region=cfg.outputs.icechunk_store_region, endpoint_url=ENDPOINT)
store.create_or_open()
report = store.validate()
print(f"Committed days: {report['committed_days']} | range {report['date_range']}")
snaps = store.list_snapshots()  # already sorted by forecast_date
DAY = snaps[-1]["forecast_date"]                       # latest real forecast day in the store
print(f"Reading forecast day: {DAY}")

session = store.readonly_session()
day_ds = xr.open_zarr(session.store, group=DAY, consolidated=False)[["tp"]]
# The virtual store carries no lat/lon arrays: rebuild them from the grid shape
# (works for both ECMWF eras, 0.25 deg = 721x1440 and 0.4 deg = 451x900).
if "latitude" not in day_ds.coords:
    dlat, dlon = lat_lon_res(day_ds.sizes["latitude"], day_ds.sizes["longitude"])
    day_ds = day_ds.assign_coords(
        latitude=90.0 - np.arange(day_ds.sizes["latitude"]) * dlat,
        longitude=np.arange(day_ds.sizes["longitude"]) * dlon)
# Subset members/steps so the live GRIB decode stays minutes-scale (real ECMWF
# data, just fewer chunks); the full pipeline in section 4 uses all 51 members.
day_ds = day_ds.sel(**BBOX).isel(member=slice(0, 6), step=slice(0, 41))
tp_mm = (day_ds["tp"] * cfg.component2.precip_scale_to_mm).load()   # m -> mm; triggers real decode
print(f"tp (bbox) shape : {tuple(tp_mm.shape)}  (member, step, lat, lon)")
print(f"members={day_ds.sizes.get('member')} steps={day_ds.sizes.get('step')} "
      f"| tp max={float(tp_mm.max()):.1f} mm")

## 2. Component 2 - real exceedance

Compute real rolling accumulations + adaptive GEV exceedance on the decoded forecast,
then read back the **live exceedance Zarr** (the persisted C2 output) from MinIO.

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations
from gik_icechain.exceedance.thresholds import (
    AdaptiveGEVThresholds, ClimateMode, classify_enso, classify_iod, get_season,
)
from gik_icechain.exceedance.exceedance import compute_exceedance_probabilities

acc = compute_rolling_accumulations(xr.Dataset({"tp": tp_mm}), windows_h=[24, 72])
thresholds = AdaptiveGEVThresholds.load(DATA / "cmorph_thresholds")
enso_iod = (pd.read_csv(DATA / "enso_iod_index.csv", parse_dates=["date"])
            .set_index("date").sort_index())
d0 = pd.Timestamp(DAY)
row = enso_iod.loc[enso_iod.index.asof(d0)]
mode = ClimateMode(get_season(d0.month),
                   classify_enso(float(row["nino34_anom"])), classify_iod(float(row["dmi"])))
thr24 = thresholds.get(24, 5, mode)
p = compute_exceedance_probabilities(acc, xr.Dataset({"rp_5y": thr24}), 24, 5, "member")
print(f"Climate mode {mode.key} | computed exceedance 24h/5yr: "
      f"max={float(p.max()):.3f} mean={float(p.mean()):.3f}")

# Live C2 output already persisted in MinIO. The store accretes days out of
# order, so sort the date index before any .sel on it.
exc_ds = xr.open_zarr(cfg.outputs.exceedance_store_uri, consolidated=False,
                      storage_options=STORAGE_OPTIONS).sortby("date")
dates = [str(x)[:10] for x in exc_ds["date"].values]
print(f"Live exceedance store: {len(dates)} dates {dates[0]}..{dates[-1]} | "
      f"windows {list(exc_ds['window'].values)} | RPs {list(exc_ds['return_period'].values)}")

## 3. Component 3 - real CRMA risk

Build the BN and infer admin-1 risk from the **live exceedance store** for one of its
real dates (max-aggregated to each unit's bbox).

In [ ]:
import geopandas as gpd
from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

model = CRMAModel(crma_cfg=cfg.component3.crma_model)
model.build()
admin = gpd.read_file(DATA / "admin_boundaries" / "east_africa_admin1.geojson")

target = pd.Timestamp(dates[-1])
exc_day = exc_ds.sel(date=target, method="nearest")
lat_dim, lon_dim = cfg.component2.spatial.lat_dim, cfg.component2.spatial.lon_dim
lat_desc = float(exc_day[lat_dim][0]) > float(exc_day[lat_dim][-1])
rows = []
for _, unit in admin.iterrows():
    b = unit.geometry.bounds  # (minx, miny, maxx, maxy)
    lat_slice = slice(b[3], b[1]) if lat_desc else slice(b[1], b[3])
    sub = exc_day["exceedance_prob"].sel({lat_dim: lat_slice, lon_dim: slice(b[0], b[2])})
    if sub.sizes.get(lat_dim, 0) == 0 or sub.sizes.get(lon_dim, 0) == 0:
        continue
    p24 = float(sub.sel(window=24, return_period=5).max())
    p72 = float(sub.sel(window=72, return_period=5).max())
    ev = CRMAEvidence(exceedance_prob_24h=p24, exceedance_prob_72h=p72, exceedance_prob_7d=p72,
                      gpm_obs_24h=0.0, api_mm=20.0,
                      spatial_coverage_fraction=float((sub.sel(window=24, return_period=5) > 0.15).mean()),
                      consecutive_signal_days=1, sat_consecutive_days=0)
    rows.append({"pcode": unit.get("admin1_pcode"), "risk": model.infer(ev)["risk_label"]})
risk_df = pd.DataFrame(rows)
if risk_df.empty:
    raise RuntimeError("No admin-1 unit intersected the exceedance grid - check bbox/orientation.")
print(f"C3 risk for {str(target)[:10]} - {len(risk_df)} units:")
print(risk_df["risk"].value_counts().to_string())

In [ ]:
colors = {"Green": "#2ecc71", "Yellow": "#f1c40f", "Orange": "#e67e22", "Red": "#e74c3c"}
counts = risk_df["risk"].value_counts().reindex(["Green", "Yellow", "Orange", "Red"], fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
ax.set_title(f"Live CRMA risk distribution - {str(target)[:10]}"); ax.set_ylabel("admin-1 units")
fig.tight_layout(); plt.show()


## 4. Full pipeline - `gik-icechain run-all` against MinIO

Runs C1 → C2 → C3 for one real day and writes `*_risk_scores.json`. This re-reads
ECMWF S3 + writes the MinIO stores, so it takes several minutes.

In [ ]:
if not RUN_FULL:
    print("Skipped (NB_RUN_FULL=0).")
else:
    OUT = REPO / "results" / "nb_live"
    cmd = [sys.executable, "-m", "gik_icechain", "run-all",
           "--start", DAY, "--end", DAY,
           "--config", str(REPO / "configs" / "default.yaml"), "--output", str(OUT)]
    print("Running:", " ".join(cmd))
    rc = subprocess.run(cmd, env=os.environ).returncode
    print("Pipeline OK" if rc == 0 else f"Pipeline failed (exit {rc})")

    import glob, json
    files = sorted(glob.glob(str(OUT / "admin1_risk" / "*_risk_scores.json")))
    for f in files:
        doc = json.load(open(f))
        c = {}
        for u in doc["units"].values():
            c[u["risk_label"]] = c.get(u["risk_label"], 0) + 1
        print(f"{doc['date']}: {len(doc['units'])} units -> " +
              ", ".join(f"{k}: {v}" for k, v in sorted(c.items())))

## 5. Validation metrics (EM-DAT)

Reproduce the EM-DAT validation on the April-2024 compound-flood window. C3 risk-only
reads the **live exceedance Zarr** (fast - no GRIB decode), then `validate-emdat`
reports unit-day AUC + **event-level early-detection** recall (the operational metric).

In [ ]:
if not RUN_FULL:
    print("Skipped (NB_RUN_FULL=0).")
else:
    VAL_START, VAL_END = "2024-04-22", "2024-04-28"   # in the live exceedance store
    VAL_OUT = REPO / "results" / "admin1_risk"
    tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
    # CHIRPS observed rainfall for the C3 API node (best-effort).
    subprocess.run([*tools, "download-gpm", "--source", "chirps",
                    "--start", "2024-04-08", "--end", VAL_END], env=os.environ, check=False)
    # C3 risk over the flood window from the live exceedance store (MinIO via AWS_ENDPOINT_URL).
    subprocess.run([sys.executable, "-m", "gik_icechain", "risk",
                    "--exceedance-store", cfg.outputs.exceedance_store_uri,
                    "--output", str(VAL_OUT), "--start", VAL_START, "--end", VAL_END,
                    "--config", str(REPO / "configs" / "default.yaml")], env=os.environ, check=True)
    # EM-DAT validation (unit-day + event-level early detection).
    subprocess.run([*tools, "validate-emdat", "--risk-dir", str(VAL_OUT),
                    "--start", VAL_START, "--end", VAL_END,
                    "--event-level", "--lead-days", "1"], env=os.environ, check=False)

## 6. Benchmarks - live store

Measures time-to-first-byte and full-scan against the live IceChunk store and writes a
CSV to `results/benchmarks/`.

In [ ]:
if not RUN_FULL:
    print("Skipped (NB_RUN_FULL=0).")
else:
    from gik_icechain.conversion.benchmark import run_benchmark

    RESULTS_DIR = REPO / "results" / "benchmarks"
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

    # Without --dynamical-store the dynamical.org row is not measured, and its 242 TB
    # stays a figure quoted by dynamical.org rather than something we measured.
    results = run_benchmark(gik_store_uri=cfg.outputs.icechunk_store_uri, domain="east_africa",
                            n_days=3, n_workers=4, output_dir=str(RESULTS_DIR))
    for name, r in results.items():
        print(f"{name}: TTFB={r.time_to_first_byte_s:.2f}s  scan={r.full_scan_elapsed_s:.1f}s  "
              f"store={r.store_size_gb:.1f} GB")

    gik = results.get("GIK+IceChunk")
    if gik:
        DYNAMICAL_FULL_GB = 242_000.0
        ratio_pct = gik.store_size_gb / DYNAMICAL_FULL_GB * 100
        checks = [
            ("time-to-first-byte < 3 s", gik.time_to_first_byte_s, "s", gik.time_to_first_byte_s < 3),
            ("storage ratio < 0.005 %", ratio_pct, "%", ratio_pct < 0.005),
        ]
        print()
        for label, value, unit, ok in checks:
            verdict = "PASS" if ok else "FAIL"
            print(f"[{verdict}] {label:26s} measured: {value:.4g} {unit}")
        print()
        print("The '< 8 h full scan on 32 vCPU' target is NOT tested here (3 days, 4 workers).")

## 7. Archive coverage - what the public bucket actually holds

No credentials needed. The archive is **not** uniform: `s3://ecmwf-forecasts` goes back
to **2023-01-18**, but the layout and the grid change partway through.

| period | prefix | grid |
|---|---|---|
| 2023-01-18 -> ~2024-02 | `{date}/00z/0p4-beta/enfo/` | 0.4 deg (451 x 900) |
| ~2024-02 -> today | `{date}/00z/ifs/0p25/enfo/` | 0.25 deg (721 x 1440) |

So the full ~1200-day corpus **is** reachable (as ICPAC's demo6 IceChunk virtual
dataset shows). The real cost of using it is not retention, it is the **grid change**:
C1 already detects `0p4-beta` (451x900), and C2 now derives the bbox slices from each
GRIB message's own Nj/Ni. Before that fix, 0.4-deg data cropped on 0.25-deg indices
returned **-17 degN when the bbox asked for 23 degN** - the wrong region, silently.

In [ ]:
import urllib.request

def _exists(url):
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=20) as r:
            return r.status == 200
    except Exception:
        return False

BASE = "https://ecmwf-forecasts.s3.amazonaws.com"
ifs_025 = lambda d: f"{BASE}/{d}/00z/ifs/0p25/enfo/{d}000000-6h-enfo-ef.grib2"
ifs_04 = lambda d: f"{BASE}/{d}/00z/0p4-beta/enfo/{d}000000-6h-enfo-ef.grib2"
aifs = lambda d: f"{BASE}/{d}/00z/aifs-ens/0p25/enfo/{d}000000-6h-enfo-pf.grib2"

print("      date   0p4-beta   0p25   AIFS")
for d in ["20230118", "20230801", "20240101", "20240301", "20241115", "20250705", "20251119"]:
    a = "yes" if _exists(ifs_04(d)) else "-"
    b = "yes" if _exists(ifs_025(d)) else "-"
    c = "yes" if _exists(aifs(d)) else "-"
    print(f"  {d}   {a:>8}   {b:>4}   {c:>4}")

print()
print("IFS ENS   2023-01-18 -> today, but 0.4 deg before ~2024-02 and 0.25 deg after")
print("AIFS ENS  from ~2025-07-05  =>  AIFS/IFS overlap ~370 days, all four seasons")
print()
print("The ~1200-day corpus is reachable. The blocker is the grid change, not retention:")
print("slicing 0.4-deg data on 0.25-deg indices silently returns the wrong region.")


## 8. Innovation 4 - AIFS vs IFS parallel track

The chain is already wired end to end in `run-all` (convert AIFS -> exceedance AIFS ->
`compute_aifs_ifs_delta` -> `seasonal_comparison`); only `aifs_track.enabled` gated it,
so `--aifs` is enough and no extra config file is needed.

Opt-in (`NB_RUN_AIFS=1`): one day, as a smoke test. The residual risk is that
`aifs_to_virtual_dataset` still uses kerchunk rather than VirtualiZarr - if anything
breaks, it breaks at the AIFS convert step.

In [ ]:
RUN_AIFS = os.getenv("NB_RUN_AIFS") == "1"
AIFS_DAY = "2025-11-19"   # inside the AIFS/IFS overlap, wet OND season
AIFS_OUT = REPO / "results" / "nb_aifs"

if not RUN_AIFS:
    print("Skipped. Set NB_RUN_AIFS=1 to run:")
    print(f"  python -m gik_icechain run-all --start {AIFS_DAY} --end {AIFS_DAY} "
          f"--aifs --output results/nb_aifs/")
else:
    cmd = [sys.executable, "-m", "gik_icechain", "run-all",
           "--start", AIFS_DAY, "--end", AIFS_DAY, "--aifs", "--output", str(AIFS_OUT)]
    print("Running:", " ".join(cmd))
    rc = subprocess.run(cmd, env=os.environ).returncode
    print("AIFS track OK" if rc == 0 else f"AIFS track failed (exit {rc})")

    import xarray as xr

    delta_path = REPO / cfg.aifs_track.comparison_output_dir / "aifs_ifs_delta.zarr"
    if delta_path.exists():
        d = xr.open_zarr(delta_path, consolidated=False)["delta_prob"]
        wetter = float((d > 0).mean()) * 100
        print(f"delta_prob (AIFS - IFS): mean={float(d.mean()):+.4f}, "
              f"AIFS wetter on {wetter:.1f}% of cells")
        print("positive => AIFS forecasts a higher exceedance probability than IFS")
    else:
        print(f"No comparison written at {delta_path}")


## 9. Innovation 2 - adaptive vs static GEV thresholds

The proposal promises an **AUC-ROC comparison of regime-aware GEV against static
thresholds**. It was never run, and until now there was no code to build the static
baseline. `build-thresholds-gpm` now takes `--pool-seasons`:

| arm | flags | varies per cell |
|---|---|---|
| adaptive | *(defaults)* | season x ENSO x IOD |
| season-only | `--min-years 999` | season |
| static | `--min-years 999 --pool-seasons` | nothing (annual maxima) |

All arms write identical filenames, so `run-all --thresholds-dir` switches between them
with no code change.

**Order matters:** regenerate the thresholds *after* the OND fix (December used to be
binned into DJF). Scoring the ablation on the old files would validate a broken season
split. The window is fixed to Nov 2024 because `satellite_validation.py` hardcodes it,
and that panel is the only ground truth with genuine dry-unit negatives.

In [ ]:
RUN_ABLATION = os.getenv("NB_RUN_ABLATION") == "1"

STEPS = """
# 0. GPM IMERG archive (needs Earthdata credentials)
python scripts/tools.py download-gpm --source nasa --start 2001-01-01 --end 2023-12-31

# 1. ADAPTIVE arm
python scripts/tools.py build-thresholds-gpm --start 2001-01-01 --end 2023-12-31 \
  --output data/thresholds_adaptive/

# 2. STATIC arm (the baseline that did not exist)
python scripts/tools.py build-thresholds-gpm --start 2001-01-01 --end 2023-12-31 \
  --output data/thresholds_static/ --min-years 999 --pool-seasons

# 3. Same window, same everything; only the threshold arm differs
python -m gik_icechain run-all --start 2024-10-31 --end 2024-11-30 \
  --thresholds-dir data/thresholds_adaptive/ \
  --exceedance-store s3://<bucket>/exc-abl-adaptive \
  --risk-output results/abl_adaptive/admin1_risk

python -m gik_icechain run-all --start 2024-10-31 --end 2024-11-30 \
  --thresholds-dir data/thresholds_static/ \
  --exceedance-store s3://<bucket>/exc-abl-static \
  --risk-output results/abl_static/admin1_risk

# 4. Score both arms on the same satellite panel
python scripts/satellite_validation.py --risk-dir results/abl_adaptive/admin1_risk
python scripts/satellite_validation.py --risk-dir results/abl_static/admin1_risk
"""

if not RUN_ABLATION:
    print("Skipped (hours of compute + the GPM archive). Set NB_RUN_ABLATION=1 to run.")
    print(STEPS)
else:
    for arm in ("adaptive", "static"):
        risk_dir = REPO / "results" / f"abl_{arm}" / "admin1_risk"
        if not risk_dir.exists():
            print(f"[{arm}] {risk_dir} missing - run the steps above first.")
            continue
        rc = subprocess.run(
            [sys.executable, str(REPO / "scripts" / "satellite_validation.py"),
             "--risk-dir", str(risk_dir)],
            env=os.environ,
        ).returncode
        print(f"[{arm}] satellite_validation exit={rc}")

print()
print("Report the result either way: soft-evidence already shipped default-OFF after it")
print("measured net-conservative. An ablation that shows no gain is a finding.")
